# TravelTide Customer Segmentation Project
## 03_FE_core_features.ipynb

**Author:** Alberto Diaz Durana  
**Date:** October 2025  
**Purpose:** Core feature engineering for customer segmentation

---

## Objectives

This notebook consolidates Week 2 Days 1-2 work to engineer core behavioral and financial features.

**Mission:** Transform raw user data into meaningful features that capture booking patterns, engagement, value, travel style, and discount sensitivity.

## Business Context

**From Notebook 02:** We have 5,765 qualified users with 41 raw features from session aggregation.

**Now:** Engineer actionable features that will power customer segmentation and enable personalized perk assignment.

**Feature Categories:**
1. **Booking Patterns:** Flight/hotel preferences, package behavior
2. **Engagement:** Session frequency, recency, activity levels
3. **Financial:** Spending patterns, transaction value, CLV segments
4. **Travel Style:** Trip characteristics, party size, duration preferences
5. **Discount Sensitivity:** Discount usage, price sensitivity indicators

**Deliverables:**
- Engineered feature dataset with 60+ features
- Feature dictionary documenting all variables
- Quality validation report
- Visualization of key feature distributions

**Outputs:**
- `../data/processed/user_features_raw.csv`
- `../data/results/feature_engineering/feature_dictionary_week2.csv`
- `../outputs/figures/feature_engineering/` (visualizations)

---

## 1. Setup & Load User Base

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Path constants
DATA_PROCESSED = '../data/processed/'
DATA_RESULTS_FE = '../data/results/feature_engineering/'
FIGURES_FE = '../outputs/figures/feature_engineering/'

# Ensure directories exist
import os
os.makedirs(DATA_RESULTS_FE, exist_ok=True)
os.makedirs(FIGURES_FE, exist_ok=True)

print(f"03_FE_core_features.ipynb - {datetime.now().strftime('%Y-%m-%d')}")
print("="*80)
print("OK: Environment configured")

03_FE_core_features.ipynb - 2025-10-27
OK: Environment configured


In [2]:
print("LOADING USER BASE FROM NOTEBOOK 02")
print("="*80)

# Load user base created in notebook 02
user_base = pd.read_csv(f'{DATA_PROCESSED}user_base_complete.csv')

# Convert date columns
date_columns = ['birthdate', 'sign_up_date']
for col in date_columns:
    if col in user_base.columns:
        user_base[col] = pd.to_datetime(user_base[col])

print(f"\nOK: User base loaded")
print(f"  Shape: {user_base.shape}")
print(f"  Users: {len(user_base):,}")
print(f"  Features: {len(user_base.columns)}")

# Display current features
print("\nCurrent features:")
print(f"  {list(user_base.columns)}")

# Quick validation
print("\nData Validation:")
print(f"  Missing user_id: {user_base['user_id'].isna().sum()}")
print(f"  Duplicate user_id: {user_base['user_id'].duplicated().sum()}")
print(f"  Age range: {user_base['age'].min():.0f}-{user_base['age'].max():.0f} years")
print(f"  Total spend range: ${user_base['total_spend'].min():.2f}-${user_base['total_spend'].max():.2f}")

print("\nOK: Data loaded and validated")

LOADING USER BASE FROM NOTEBOOK 02

OK: User base loaded
  Shape: (5765, 41)
  Users: 5,765
  Features: 41

Current features:
  ['user_id', 'birthdate', 'gender', 'married', 'has_children', 'home_country', 'home_city', 'home_airport', 'sign_up_date', 'age', 'total_sessions', 'total_page_clicks', 'avg_page_clicks_per_session', 'avg_session_duration_minutes', 'median_session_duration_minutes', 'total_flights_booked', 'total_hotels_booked', 'total_cancellations', 'total_package_bookings', 'total_flight_spend', 'avg_flight_fare', 'total_seats_purchased', 'total_bags', 'avg_bags_per_trip', 'return_flight_count', 'total_hotel_spend', 'avg_hotel_price_per_night', 'total_hotel_nights', 'avg_nights_per_stay', 'total_rooms_booked', 'avg_rooms_per_booking', 'days_since_signup', 'years_active', 'flight_trips', 'hotel_trips', 'total_spend', 'cancellation_rate', 'years_active_adjusted', 'estimated_annual_clv', 'clv_quartile', 'clv_segment']

Data Validation:
  Missing user_id: 0
  Duplicate user_id:

## 2. Booking Pattern Features

Engineer features that capture flight vs hotel preferences, package behavior, and booking channel diversity.

In [3]:
print("ENGINEERING BOOKING PATTERN FEATURES")
print("="*80)

# Calculate total bookings
user_base['total_all_bookings'] = (
    user_base['total_flights_booked'] + 
    user_base['total_hotels_booked']
)

# Booking rates (what proportion of their bookings are each type)
user_base['flight_only_bookings'] = user_base['total_flights_booked'] - user_base['total_package_bookings']
user_base['hotel_only_bookings'] = user_base['total_hotels_booked'] - user_base['total_package_bookings']

# Booking type rates
user_base['flight_only_rate'] = np.where(
    user_base['total_all_bookings'] > 0,
    user_base['flight_only_bookings'] / user_base['total_all_bookings'],
    0
)

user_base['hotel_only_rate'] = np.where(
    user_base['total_all_bookings'] > 0,
    user_base['hotel_only_bookings'] / user_base['total_all_bookings'],
    0
)

user_base['package_booking_rate'] = np.where(
    user_base['total_all_bookings'] > 0,
    user_base['total_package_bookings'] / user_base['total_all_bookings'],
    0
)

# Hotel booking rate (proportion of sessions that included hotel)
user_base['hotel_booking_rate'] = user_base['total_hotels_booked'] / user_base['total_sessions']

# Channel diversity (do they use both flight and hotel, or just one?)
user_base['uses_both_channels'] = (
    (user_base['total_flights_booked'] > 0) & 
    (user_base['total_hotels_booked'] > 0)
)

user_base['channel_diversity_score'] = (
    user_base['uses_both_channels'].astype(int) * 
    np.minimum(user_base['total_flights_booked'], user_base['total_hotels_booked'])
)

# Preferred channel
def get_preferred_channel(row):
    if row['total_flights_booked'] == 0 and row['total_hotels_booked'] == 0:
        return 'None'
    elif row['total_flights_booked'] > row['total_hotels_booked']:
        return 'Flight'
    elif row['total_hotels_booked'] > row['total_flights_booked']:
        return 'Hotel'
    else:
        return 'Balanced'

user_base['preferred_channel'] = user_base.apply(get_preferred_channel, axis=1)

# Binary flags for dominant booking types
user_base['is_flight_focused'] = user_base['flight_only_rate'] > 0.5
user_base['is_hotel_focused'] = user_base['hotel_only_rate'] > 0.5
user_base['is_package_traveler'] = user_base['package_booking_rate'] > 0.5
user_base['is_hotel_only_traveler'] = (
    (user_base['total_hotels_booked'] > 0) & 
    (user_base['total_flights_booked'] == 0)
)

print(f"\nOK: Booking pattern features created")
print(f"\nBooking Pattern Summary:")
print("-" * 80)
print(f"  Avg bookings per user: {user_base['total_all_bookings'].mean():.2f}")
print(f"  Flight-only rate: {user_base['flight_only_rate'].mean():.2%}")
print(f"  Hotel-only rate: {user_base['hotel_only_rate'].mean():.2%}")
print(f"  Package rate: {user_base['package_booking_rate'].mean():.2%}")
print(f"  Users with both channels: {user_base['uses_both_channels'].sum():,} ({user_base['uses_both_channels'].sum()/len(user_base)*100:.1f}%)")

print(f"\nPreferred Channel Distribution:")
for channel, count in user_base['preferred_channel'].value_counts().items():
    print(f"  {channel}: {count:,} ({count/len(user_base)*100:.1f}%)")

print("\nOK: Booking pattern features complete")

ENGINEERING BOOKING PATTERN FEATURES

OK: Booking pattern features created

Booking Pattern Summary:
--------------------------------------------------------------------------------
  Avg bookings per user: 3.84
  Flight-only rate: 7.44%
  Hotel-only rate: 9.47%
  Package rate: 35.27%
  Users with both channels: 4,581 (79.5%)

Preferred Channel Distribution:
  Balanced: 3,016 (52.3%)
  Hotel: 1,061 (18.4%)
  Flight: 965 (16.7%)
  None: 723 (12.5%)

OK: Booking pattern features complete


## 3. Engagement & Activity Features

Engineer features capturing session frequency, recency, activity patterns, and conversion behavior.

In [4]:
print("ENGINEERING ENGAGEMENT & ACTIVITY FEATURES")
print("="*80)

# Reference date for recency calculations
reference_date = pd.Timestamp('2023-04-30')

# Sessions per month (activity intensity)
user_base['active_months'] = (user_base['years_active'] * 12).apply(lambda x: max(x, 1))
user_base['sessions_per_month'] = user_base['total_sessions'] / user_base['active_months']

# Days since last activity (would need session dates - using placeholder logic)
# In real implementation, this would calculate from max(session_date)
# For now, we'll create a proxy based on tenure
user_base['days_since_last_session'] = (
    user_base['days_since_signup'] * 0.1  # Proxy: assume recent users more active
).astype(int)

user_base['days_since_last_booking'] = (
    user_base['days_since_signup'] * 0.15  # Proxy: slightly longer than last session
)

# Booking velocity (bookings per month)
user_base['booking_velocity'] = user_base['total_all_bookings'] / user_base['active_months']

# Browse-to-book ratio (how many sessions before booking)
user_base['browse_to_book_ratio'] = np.where(
    user_base['total_all_bookings'] > 0,
    user_base['total_sessions'] / user_base['total_all_bookings'],
    user_base['total_sessions']  # All browsing, no booking
)

# Booking conversion rate
user_base['booking_conversion_rate'] = user_base['total_all_bookings'] / user_base['total_sessions']

# Average days between bookings
user_base['avg_days_between_bookings'] = np.where(
    user_base['total_all_bookings'] > 1,
    user_base['days_since_signup'] / (user_base['total_all_bookings'] - 1),
    user_base['days_since_signup']  # Only one booking or none
)

# Activity flags
user_base['is_active_booker'] = (
    (user_base['total_all_bookings'] > 0) & 
    (user_base['days_since_last_booking'] < 90)
)

# Cancellation metrics
user_base['cancellation_frequency'] = user_base['total_cancellations'] / user_base['total_sessions']
user_base['has_recent_cancellation'] = user_base['total_cancellations'] > 0

# Days since last cancellation (only for users with cancellations)
user_base['days_since_last_cancellation'] = np.where(
    user_base['total_cancellations'] > 0,
    user_base['days_since_signup'] * 0.2,  # Proxy
    np.nan
)

print(f"\nOK: Engagement features created")
print(f"\nEngagement Summary:")
print("-" * 80)
print(f"  Avg sessions per month: {user_base['sessions_per_month'].mean():.2f}")
print(f"  Avg booking velocity: {user_base['booking_velocity'].mean():.2f} bookings/month")
print(f"  Avg browse-to-book ratio: {user_base['browse_to_book_ratio'].mean():.2f} sessions per booking")
print(f"  Avg conversion rate: {user_base['booking_conversion_rate'].mean():.2%}")
print(f"  Active bookers: {user_base['is_active_booker'].sum():,} ({user_base['is_active_booker'].sum()/len(user_base)*100:.1f}%)")
print(f"  Users with cancellations: {user_base['has_recent_cancellation'].sum():,} ({user_base['has_recent_cancellation'].sum()/len(user_base)*100:.1f}%)")

print("\nOK: Engagement & activity features complete")

ENGINEERING ENGAGEMENT & ACTIVITY FEATURES

OK: Engagement features created

Engagement Summary:
--------------------------------------------------------------------------------
  Avg sessions per month: 2.53
  Avg booking velocity: 1.27 bookings/month
  Avg browse-to-book ratio: 3.06 sessions per booking
  Avg conversion rate: 51.55%
  Active bookers: 5,041 (87.4%)
  Users with cancellations: 0 (0.0%)

OK: Engagement & activity features complete


## 4. Financial & Value Features

Engineer features capturing spending patterns, transaction value, and customer value segments.

In [ ]:
print("ENGINEERING FINANCIAL & VALUE FEATURES")
print("="*80)

# Average transaction value
user_base['avg_transaction_value'] = np.where(
    user_base['total_all_bookings'] > 0,
    user_base['total_spend'] / user_base['total_all_bookings'],
    0
)

# Flight vs hotel transaction averages
user_base['flight_transaction_avg'] = np.where(
    user_base['total_flights_booked'] > 0,
    user_base['total_flight_spend'] / user_base['total_flights_booked'],
    0
)

user_base['hotel_transaction_avg'] = np.where(
    user_base['total_hotels_booked'] > 0,
    user_base['total_hotel_spend'] / user_base['total_hotels_booked'],
    0
)

# Spending consistency (coefficient of variation - would need individual transactions)
# Using proxy: standard deviation of flight vs hotel spending patterns
user_base['spending_consistency_cv'] = np.where(
    (user_base['total_flights_booked'] > 0) & (user_base['total_hotels_booked'] > 0),
    np.abs(user_base['avg_flight_fare'] - user_base['avg_hotel_price_per_night']) / 
    ((user_base['avg_flight_fare'] + user_base['avg_hotel_price_per_night']) / 2),
    0
)

# High value customer flag
clv_75th_percentile = user_base['estimated_annual_clv'].quantile(0.75)
user_base['is_high_value_customer'] = user_base['estimated_annual_clv'] >= clv_75th_percentile

# CLV segments (already have, but confirm)
print(f"\nCLV Segmentation:")
print("-" * 80)
for segment in ['Low Value', 'Medium Value', 'High Value', 'VIP']:
    count = (user_base['clv_segment'] == segment).sum()
    avg_clv = user_base[user_base['clv_segment'] == segment]['estimated_annual_clv'].mean()
    print(f"  {segment}: {count:,} ({count/len(user_base)*100:.1f}%) | Avg CLV: ${avg_clv:.2f}")

print(f"\nFinancial Metrics Summary:")
print("-" * 80)
print(f"  Avg transaction value: ${user_base['avg_transaction_value'].mean():.2f}")
print(f"  Avg flight transaction: ${user_base['flight_transaction_avg'].mean():.2f}")
print(f"  Avg hotel transaction: ${user_base['hotel_transaction_avg'].mean():.2f}")
print(f"  High value customers: {user_base['is_high_value_customer'].sum():,} ({user_base['is_high_value_customer'].sum()/len(user_base)*100:.1f}%)")

print("\nOK: Financial & value features complete")